# 109. Video Analysis: Understanding Video Content

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/13-multi-modal/109_video_analysis.ipynb)

**Category:** 13 - Multi-Modal Techniques  
**Technique #:** 109  
**Difficulty:** Advanced

## 📖 Description

Video Analysis involves extracting frames from video content and analyzing them using vision-language models. This technique enables understanding of temporal content, actions, and narratives across video sequences.

### When to Use:
- Content moderation for videos
- Video summarization and indexing
- Action recognition and tracking
- Educational video analysis
- Surveillance and security monitoring

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                   VIDEO ANALYSIS FLOW                        │
└─────────────────────────────────────────────────────────────┘

    ┌──────────────┐         ┌──────────────┐         ┌──────────────┐
    │   Video      │────────▶│   Frame      │────────▶│   Frame      │
    │   Input      │         │   Extraction│         │   Analysis   │
    └──────────────┘         └──────────────┘         └──────┬───────┘
         (mp4, mov)            (sampled frames)              │
                                                             ▼
                                                      ┌──────────────┐
                                                      │  Temporal    │
                                                      │  Aggregation │
                                                      └──────┬───────┘
                                                             │
                                                             ▼
                                                      ┌──────────────┐
                                                      │   Video      │
                                                      │   Summary    │
                                                      └──────────────┘
```

### Key Approaches:
- **Frame Sampling**: Extract key frames at intervals
- **Scene Detection**: Identify scene changes
- **Temporal Analysis**: Track changes across frames
- **Audio-Visual Fusion**: Combine visual and audio analysis

## 🛠️ Setup

In [ ]:
!pip install -q openai pillow requests opencv-python-headless

In [ ]:
import os
from getpass import getpass
import base64
import requests
import cv2
import io
from PIL import Image

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

## 💡 Basic Example

In [ ]:
def extract_frames_from_video(video_path, num_frames=5):
    """Extract evenly spaced frames from a video."""
    frames = []
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return []
    
    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            
            # Convert to base64
            buffer = io.BytesIO()
            pil_image.save(buffer, format="JPEG")
            img_str = base64.b64encode(buffer.getvalue()).decode()
            frames.append(img_str)
    
    cap.release()
    return frames

def encode_image_from_url(image_url):
    """Encode image from URL to base64."""
    response = requests.get(image_url)
    return base64.b64encode(response.content).decode('utf-8')

def analyze_video_frames(frame_base64_list, analysis_type="summary", model="gpt-4o"):
    """Analyze multiple video frames."""
    
    analysis_prompts = {
        "summary": """
        Analyze these frames from a video and provide:
        1. Overall scene description
        2. Key objects and people visible
        3. Main activities or actions
        4. Setting/environment
        5. Any text or notable visual elements
        """,
        "actions": """
        Focus on actions and movements in these video frames:
        1. What actions are people/objects performing?
        2. How do positions change between frames?
        3. What is the sequence of events?
        4. Any notable motion or activity?
        """,
        "objects": """
        Identify and track objects across these video frames:
        1. List all distinct objects visible
        2. Note which objects appear in multiple frames
        3. Describe any object interactions
        4. Track object positions if they move
        """
    }
    
    prompt = analysis_prompts.get(analysis_type, analysis_prompts["summary"])
    
    # Build content with all frames
    content = [{"type": "text", "text": prompt}]
    for frame_b64 in frame_base64_list:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{frame_b64}"
            }
        })
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": content
                }
            ],
            max_tokens=1500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Demo with sample images (simulating video frames)
print("VIDEO ANALYSIS - FRAME SEQUENCE EXAMPLE\n")
print("="*60 + "\n")

# Using sample images as "frames"
frame_urls = [
    "https://images.unsplash.com/photo-1518837695005-2083093ee35b?w=400",
    "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400",
    "https://images.unsplash.com/photo-1470071459604-3b5ec3a7fe05?w=400"
]

frames = [encode_image_from_url(url) for url in frame_urls]
result = analyze_video_frames(frames, analysis_type="summary")
print(result)

## 🌍 Real-World Example

In [ ]:
# Real-world: Content moderation for user-generated videos
def moderate_video_content(frame_base64_list):
    """Analyze video frames for content moderation."""
    
    prompt = """
    You are a content moderator. Analyze these video frames for:
    
    ## Safety Assessment
    - Violent content (yes/no, confidence: high/medium/low)
    - Adult/explicit content (yes/no, confidence)
    - Hate symbols or offensive content (yes/no, confidence)
    - Dangerous activities (yes/no, confidence)
    
    ## Content Classification
    - Primary category (education/entertainment/news/sports/other)
    - Age appropriateness (all ages/teen/adult)
    
    ## Recommendation
    - Action: (approve/flag for review/reject)
    - Confidence level
    - Reasoning
    """
    
    content = [{"type": "text", "text": prompt}]
    for frame_b64 in frame_base64_list:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{frame_b64}"
            }
        })
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": content
                }
            ],
            max_tokens=800
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Example with nature scene frames
nature_frames = [
    "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400",
    "https://images.unsplash.com/photo-1469474968028-56623f02e42e?w=400",
    "https://images.unsplash.com/photo-1447752875215-b2761acb3c5d?w=400"
]

nature_frame_b64 = [encode_image_from_url(url) for url in nature_frames]

print("CONTENT MODERATION EXAMPLE\n")
print("="*60 + "\n")

moderation_result = moderate_video_content(nature_frame_b64)
print(moderation_result)

## ❌ Failure Case

In [ ]:
# Failure case: Fast motion and frame sampling limitations
print("VIDEO ANALYSIS LIMITATIONS\n")
print("="*60 + "\n")

limitations = [
    {
        "issue": "Fast motion/events",
        "problem": "Key events may occur between sampled frames",
        "solution": "Increase frame sampling rate or use scene detection"
    },
    {
        "issue": "Temporal context",
        "problem": "Models see frames as independent images",
        "solution": "Use video-native models like Gemini 1.5 Pro"
    },
    {
        "issue": "Audio-visual correlation",
        "problem": "Visual-only analysis misses audio cues",
        "solution": "Combine with audio transcription/analysis"
    },
    {
        "issue": "Long videos",
        "problem": "Token limits restrict number of frames",
        "solution": "Use hierarchical analysis or video-specific models"
    }
]

for lim in limitations:
    print(f"⚠️  {lim['issue']}")
    print(f"   Problem: {lim['problem']}")
    print(f"   Solution: {lim['solution']}\n")

print("="*60)
print("NOTE: For production video analysis, consider:")
print("- Google Gemini 1.5 Pro (native video support)")
print("- AWS Rekognition Video")
print("- Azure Video Indexer")

## 📊 Benchmark Comparison

| Capability | Frame Sampling | Gemini 1.5 Pro | AWS Rekognition | Azure Video |
|------------|----------------|----------------|-----------------|-------------|
| Frame Analysis | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| Temporal Understanding | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| Action Recognition | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| Audio Integration | ⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| Cost Efficiency | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |

### Recommendations:
- **Frame Sampling**: Quick prototyping, budget-conscious
- **Gemini 1.5 Pro**: Best overall video understanding
- **AWS/Azure**: Enterprise, high-volume processing

## 🎮 Interactive Playground

In [ ]:
def video_analysis_playground():
    """Interactive video analysis playground."""
    print("\n" + "="*60)
    print("VIDEO ANALYSIS PLAYGROUND")
    print("="*60 + "\n")
    
    print("Enter URLs for video frames (3-5 images recommended):\n")
    
    frame_urls = []
    for i in range(5):
        url = input(f"Frame {i+1} URL (or press Enter to finish): ").strip()
        if not url:
            break
        frame_urls.append(url)
    
    if not frame_urls:
        # Default frames
        frame_urls = [
            "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400",
            "https://images.unsplash.com/photo-1469474968028-56623f02e42e?w=400",
            "https://images.unsplash.com/photo-1447752875215-b2761acb3c5d?w=400"
        ]
        print("Using default sample frames.\n")
    
    print("\nSelect analysis type:")
    print("1. General summary")
    print("2. Action/movement focus")
    print("3. Object tracking")
    print("4. Content moderation")
    
    analysis_choice = input("Enter choice (1-4): ").strip()
    
    analysis_types = {
        "1": "summary",
        "2": "actions",
        "3": "objects",
        "4": "moderation"
    }
    
    selected_type = analysis_types.get(analysis_choice, "summary")
    
    print("\nAnalyzing frames...\n")
    
    frames = [encode_image_from_url(url) for url in frame_urls]
    
    if selected_type == "moderation":
        result = moderate_video_content(frames)
    else:
        result = analyze_video_frames(frames, analysis_type=selected_type)
    
    print("="*60)
    print("ANALYSIS RESULT:")
    print("="*60)
    print(result)

video_analysis_playground()

## 💡 Tips & Tricks

### Frame Selection:
1. **Sample evenly**: Distribute frames across video duration
2. **Scene detection**: Extract frames at scene changes
3. **Key moments**: Focus on important timestamps
4. **Limit count**: 3-8 frames optimal for most models

### Analysis Strategies:
- **Hierarchical**: Analyze segments, then combine
- **Focused**: Crop to regions of interest
- **Comparative**: Compare frames for changes
- **Contextual**: Include audio transcripts

### Optimization:
- Resize frames to 512x512 or smaller
- Use JPEG compression
- Batch similar videos together
- Cache frame encodings

## 📚 References

1. [Gemini 1.5 Pro Video Understanding](https://ai.google.dev/gemini-api/docs/video-understanding)
2. [AWS Rekognition Video](https://aws.amazon.com/rekognition/video-features/)
3. [Azure Video Indexer](https://azure.microsoft.com/en-us/services/video-indexer/)
4. [OpenCV Video Processing](https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html)